# Query Iceberg Tables via DuckDB + Polaris REST Catalog

Reads `demo.users` and `demo.transactions` written by the Flink job, using
DuckDB's native Iceberg REST catalog support connected to Apache Polaris.

**Prerequisites:** the Docker Compose stack must be running (`docker compose up -d`).

In [4]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("INSTALL iceberg")
con.execute("LOAD httpfs")
con.execute("LOAD iceberg")

# S3 credentials — DuckDB uses these to sign its own S3 requests directly
con.execute("""
CREATE OR REPLACE SECRET minio (
    TYPE        s3,
    KEY_ID      'minioadmin',
    SECRET      'minioadmin',
    ENDPOINT    'minio:9000',
    URL_STYLE   'path',
    USE_SSL     false,
    REGION      'us-east-1'
)
""")

# DuckDB 1.2+ requires OAuth2 credentials in a separate TYPE iceberg secret;
# SCOPE and other auth params are no longer accepted inline in ATTACH.
con.execute("""
CREATE OR REPLACE SECRET polaris_oauth (
    TYPE              iceberg,
    CLIENT_ID         'root',
    CLIENT_SECRET     's3cr3t',
    OAUTH2_SERVER_URI 'http://polaris:8181/api/catalog/v1/oauth/tokens',
    SCOPE             'PRINCIPAL_ROLE:ALL'
)
""")

# ACCESS_DELEGATION_MODE 'none' tells DuckDB to use the minio secret above
# rather than requesting vended credentials from Polaris.
con.execute("""
ATTACH 'demo_lh' AS polaris (
    TYPE                   iceberg,
    ENDPOINT               'http://polaris:8181/api/catalog',
    SECRET                 polaris_oauth,
    ACCESS_DELEGATION_MODE 'none'
)
""")

con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,polaris,demo,transactions,[__],[UNKNOWN],False
1,polaris,demo,users,[__],[UNKNOWN],False


---
## Users

In [5]:
# All users
con.execute("SELECT * FROM polaris.demo.users ORDER BY created_at desc").df()

,user_id,name,email,country,created_at
0,user-050,Karen Smith,karen.smith50@example.com,AU,2026-05-15 17:55:50.080
1,user-001,Tara Robinson,tara.robinson1@example.com,US,2026-05-15 17:55:50.080
2,user-049,Alice Thomas,alice.thomas49@example.com,US,2026-05-15 17:55:50.080
3,user-048,Leo Clark,leo.clark48@example.com,CA,2026-05-15 17:55:50.080
4,user-047,Quinn Johnson,quinn.johnson47@example.com,US,2026-05-15 17:55:50.080
...,...,...,...,...,...
95,user-006,Olivia Lewis,olivia.lewis6@example.com,AU,2026-05-14 18:26:54.740
96,user-005,Grace Johnson,grace.johnson5@example.com,FR,2026-05-14 18:26:54.740
97,user-004,Rose Garcia,rose.garcia4@example.com,BR,2026-05-14 18:26:54.740
98,user-003,Ned Williams,ned.williams3@example.com,AU,2026-05-14 18:26:54.740


In [6]:
# Summary stats
con.execute("""
    SELECT
        count(*)                AS total_users,
        count(DISTINCT country) AS unique_countries,
        min(created_at)         AS earliest_signup,
        max(created_at)         AS latest_signup
    FROM polaris.demo.users
""").df()

,total_users,unique_countries,earliest_signup,latest_signup
0,100,8,2026-05-14 18:26:54.740,2026-05-15 17:55:50.080


In [7]:
# Users per country
con.execute("""
    SELECT
        country,
        count(*) AS user_count
    FROM polaris.demo.users
    GROUP BY country
    ORDER BY user_count DESC
""").df()

,country,user_count
0,AU,20
1,US,18
2,BR,13
3,CA,13
4,DE,12
5,JP,11
6,GB,8
7,FR,5


---
## Transactions

In [8]:
# Latest 20 transactions
con.execute("""
    SELECT *
    FROM polaris.demo.transactions
    ORDER BY event_time DESC
    LIMIT 20
""").df()

,transaction_id,user_id,amount,currency,type,status,event_time
0,txn-005880,user-019,1035.69,GBP,REFUND,COMPLETED,2026-05-15 18:45:09.093
1,txn-005879,user-017,746.79,GBP,WITHDRAWAL,FAILED,2026-05-15 18:45:08.591
2,txn-005878,user-042,640.42,AUD,DEPOSIT,PENDING,2026-05-15 18:45:08.090
3,txn-005877,user-017,148.46,CAD,TRANSFER,REVERSED,2026-05-15 18:45:07.588
4,txn-005876,user-037,1611.52,CAD,REFUND,REVERSED,2026-05-15 18:45:07.088
5,txn-005875,user-005,62.27,AUD,REFUND,COMPLETED,2026-05-15 18:45:06.586
6,txn-005874,user-011,784.56,EUR,PURCHASE,PENDING,2026-05-15 18:45:06.084
7,txn-005873,user-019,1455.82,USD,WITHDRAWAL,COMPLETED,2026-05-15 18:45:05.564
8,txn-005872,user-050,1729.08,EUR,DEPOSIT,COMPLETED,2026-05-15 18:45:05.062
9,txn-005871,user-017,907.39,EUR,REFUND,FAILED,2026-05-15 18:45:04.559


In [9]:
# Summary stats
con.execute("""
    SELECT
        count(*)                AS total_transactions,
        count(DISTINCT user_id) AS unique_users,
        round(sum(amount), 2)   AS total_volume,
        round(avg(amount), 2)   AS avg_amount,
        min(event_time)         AS earliest,
        max(event_time)         AS latest
    FROM polaris.demo.transactions
""").df()

,total_transactions,unique_users,total_volume,avg_amount,earliest,latest
0,49912,50,49842033.5,998.6,2026-05-14 18:26:55.065,2026-05-15 18:45:09.093


In [10]:
# Volume by transaction type
con.execute("""
    SELECT
        type,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM polaris.demo.transactions
    GROUP BY type
    ORDER BY tx_count DESC
""").df()

,type,tx_count,total_amount
0,PURCHASE,10126,10100225.56
1,REFUND,10029,10062378.21
2,WITHDRAWAL,10021,9971751.14
3,DEPOSIT,9875,9834708.37
4,TRANSFER,9861,9872970.22


In [11]:
# Breakdown by status
con.execute("""
    SELECT
        status,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM polaris.demo.transactions
    GROUP BY status
    ORDER BY tx_count DESC
""").df()

,status,tx_count,total_amount
0,PENDING,12728,12745331.47
1,REVERSED,12452,12392498.65
2,FAILED,12407,12457510.46
3,COMPLETED,12325,12246692.92


In [12]:
# Volume by currency
con.execute("""
    SELECT
        currency,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM polaris.demo.transactions
    GROUP BY currency
    ORDER BY total_amount DESC
""").df()

,currency,tx_count,total_amount
0,GBP,10080,10110094.21
1,EUR,10058,10041600.14
2,AUD,9941,9922785.65
3,CAD,9865,9900531.00
4,USD,9968,9867022.50


In [13]:
# Top 10 users by transaction volume
con.execute("""
    SELECT
        user_id,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_spent
    FROM polaris.demo.transactions
    GROUP BY user_id
    ORDER BY total_spent DESC
    LIMIT 10
""").df()

,user_id,tx_count,total_spent
0,user-006,1062,1077172.58
1,user-005,1047,1052663.21
2,user-037,1047,1047303.73
3,user-004,1026,1041655.98
4,user-008,1043,1038821.82
5,user-038,1038,1034905.88
6,user-003,994,1034245.36
7,user-039,1025,1033981.82
8,user-041,1037,1031464.33
9,user-001,1025,1028708.48


---
## Cross-table: join users ↔ transactions

In [14]:
# Enrich transactions with user name and country
con.execute("""
    SELECT
        t.transaction_id,
        u.name,
        u.country,
        t.type,
        t.currency,
        t.amount,
        t.status,
        t.event_time
    FROM polaris.demo.transactions AS t
    JOIN polaris.demo.users        AS u ON t.user_id = u.user_id
    ORDER BY t.event_time DESC
    LIMIT 20
""").df()

,transaction_id,name,country,type,currency,amount,status,event_time
0,txn-005880,Bob Thomas,CA,REFUND,GBP,1035.69,COMPLETED,2026-05-15 18:45:09.093
1,txn-005880,Bob Martin,AU,REFUND,GBP,1035.69,COMPLETED,2026-05-15 18:45:09.093
2,txn-005879,Tara Evans,CA,WITHDRAWAL,GBP,746.79,FAILED,2026-05-15 18:45:08.591
3,txn-005879,Olivia Davies,DE,WITHDRAWAL,GBP,746.79,FAILED,2026-05-15 18:45:08.591
4,txn-005878,Ned Evans,JP,DEPOSIT,AUD,640.42,PENDING,2026-05-15 18:45:08.090
5,txn-005878,Grace Clark,US,DEPOSIT,AUD,640.42,PENDING,2026-05-15 18:45:08.090
6,txn-005877,Olivia Davies,DE,TRANSFER,CAD,148.46,REVERSED,2026-05-15 18:45:07.588
7,txn-005877,Tara Evans,CA,TRANSFER,CAD,148.46,REVERSED,2026-05-15 18:45:07.588
8,txn-005876,Grace Thomas,GB,REFUND,CAD,1611.52,REVERSED,2026-05-15 18:45:07.088
9,txn-005876,Hank Thomas,BR,REFUND,CAD,1611.52,REVERSED,2026-05-15 18:45:07.088


In [15]:
# Total transaction volume per country
con.execute("""
    SELECT
        u.country,
        count(*)                AS tx_count,
        round(sum(t.amount), 2) AS total_volume
    FROM polaris.demo.transactions AS t
    JOIN polaris.demo.users        AS u ON t.user_id = u.user_id
    GROUP BY u.country
    ORDER BY total_volume DESC
""").df()

,country,tx_count,total_volume
0,AU,20020,19892022.71
1,US,18077,17958113.73
2,BR,13155,13171348.31
3,CA,12767,12795754.21
4,DE,11856,11824404.27
5,JP,10930,10871926.17
6,GB,8003,8093050.37
7,FR,5016,5077447.23


---
## Snapshot history

In [16]:
# Current snapshot info via DuckDB iceberg_snapshots()
for table_name in ("users", "transactions"):
    snap_df = con.execute(f"""
        SELECT snapshot_id, timestamp_ms, manifest_list
        FROM iceberg_snapshots('polaris.demo.{table_name}')
        ORDER BY timestamp_ms DESC
        LIMIT 1
    """).df()
    if not snap_df.empty:
        row = snap_df.iloc[0]
        print(f"=== {table_name} === snapshot_id={row.snapshot_id}  manifest_list={row.manifest_list}")
    else:
        print(f"=== {table_name} === no snapshots found")

=== users === snapshot_id=6668326028793722795  manifest_list=s3://iceberg-warehouse/demo/users/metadata/snap-6668326028793722795-1-99e7dc9c-0096-423b-85d6-09905a98c48c.avro
=== transactions === snapshot_id=435431072021520727  manifest_list=s3://iceberg-warehouse/demo/transactions/metadata/snap-435431072021520727-1-ae53ce67-8346-46a6-8b0f-c1be26dfc6be.avro
